# 04 - Model Training & Evaluation
## RailOpt AI: Railway Maintenance Priority Prediction

## Section 1: Project Introduction

Several regression algorithms are trained because railway priority patterns may be captured differently by bagging, boosting, and randomized tree ensembles. Comparing candidates on the same held-out validation set makes their trade-offs measurable rather than relying on model-family assumptions.

The selection metric is **validation R²**, with lower validation RMSE used as a tie-breaker. MAE, MSE, and RMSE also show error magnitude in priority-score units. Five-fold cross-validation on the training partition measures stability. The test partition remains reserved for the final selected model. This notebook does not perform scheduling or OR-Tools optimization.

---
## Section 2: Import Libraries
Import data, plotting, persistence, metrics, validation, and model libraries used in this training stage.

In [1]:
from pathlib import Path
from time import perf_counter
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from lightgbm import LGBMRegressor
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, RandomizedSearchCV, cross_val_score
from xgboost import XGBRegressor

warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid", context="notebook")

current_dir = Path.cwd()
PROJECT_ROOT = current_dir if (current_dir / "data").exists() else current_dir.parent
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "outputs" / "models"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
REPORTS_DIR = PROJECT_ROOT / "outputs" / "reports"
for output_dir in (MODELS_DIR, FIGURES_DIR, REPORTS_DIR):
    output_dir.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
TARGET = "priority_score"
print(f"Project root: {PROJECT_ROOT}")
print(f"XGBoost: {XGBRegressor.__module__}; LightGBM: {LGBMRegressor.__module__}")

ImportError: DLL load failed while importing _gradient_boosting: An Application Control policy has blocked this file.

---
## Section 3: Load Processed Dataset
Load the three Stage 3 splits, check their dimensions and schema consistency, and identify the regression target.

In [ ]:
split_paths = {
    "train": PROCESSED_DATA_DIR / "train.csv",
    "validation": PROCESSED_DATA_DIR / "validation.csv",
    "test": PROCESSED_DATA_DIR / "test.csv",
}
missing_files = [str(path) for path in split_paths.values() if not path.exists()]
if missing_files:
    raise FileNotFoundError(f"Processed dataset files not found: {missing_files}")

datasets = {name: pd.read_csv(path) for name, path in split_paths.items()}
for split_name, frame in datasets.items():
    print(f"{split_name.title():<10}: {frame.shape[0]:,} rows x {frame.shape[1]} columns")

train_columns = list(datasets["train"].columns)
if TARGET not in train_columns:
    raise KeyError(f"Target column '{TARGET}' is missing from train.csv.")
for split_name, frame in datasets.items():
    if frame.empty:
        raise ValueError(f"The {split_name} dataset is empty.")
    if list(frame.columns) != train_columns:
        raise ValueError(f"Column names or order differ in {split_name}.csv.")
    if frame.isna().any().any():
        raise ValueError(f"Missing values found in {split_name}.csv.")

print(f"Target variable: {TARGET}")
print(f"Feature count: {len(train_columns) - 1}")
print("Stage 3 scaling is already applied to numeric features; encoded categories remain unscaled.")

---
## Section 4: Separate Features and Target
Use every processed column except `priority_score` as a feature. Check that all model inputs are numeric, as expected from Stage 3 preprocessing.

In [ ]:
X_train = datasets["train"].drop(columns=TARGET)
y_train = datasets["train"][TARGET]
X_validation = datasets["validation"].drop(columns=TARGET)
y_validation = datasets["validation"][TARGET]
X_test = datasets["test"].drop(columns=TARGET)
y_test = datasets["test"][TARGET]
feature_columns = X_train.columns.tolist()

if not all(pd.api.types.is_numeric_dtype(dtype) for dtype in X_train.dtypes):
    non_numeric = X_train.select_dtypes(exclude=np.number).columns.tolist()
    raise TypeError(f"Non-numeric model features found: {non_numeric}")
if not X_train.columns.equals(X_validation.columns) or not X_train.columns.equals(X_test.columns):
    raise ValueError("Feature columns do not match across the three splits.")

print(f"Training features: {X_train.shape}; target: {y_train.shape}")
print(f"Validation features: {X_validation.shape}; target: {y_validation.shape}")
print(f"Test features: {X_test.shape}; target: {y_test.shape}")

---
## Section 5: Train Models
Fit five complementary tree-based regressors on the training split only. Record wall-clock fit time and retain validation predictions for a like-for-like comparison.

In [ ]:
def build_models():
    """Return reproducible baseline regressors with practical defaults."""
    return {
        "Random Forest": RandomForestRegressor(
            n_estimators=250, min_samples_leaf=2, n_jobs=-1, random_state=RANDOM_STATE
        ),
        "XGBoost": XGBRegressor(
            n_estimators=250, max_depth=6, learning_rate=0.05,
            subsample=0.9, colsample_bytree=0.9, objective="reg:squarederror",
            n_jobs=-1, random_state=RANDOM_STATE, verbosity=0
        ),
        "LightGBM": LGBMRegressor(
            n_estimators=250, learning_rate=0.05, num_leaves=31,
            n_jobs=-1, random_state=RANDOM_STATE, verbosity=-1
        ),
        "Gradient Boosting": GradientBoostingRegressor(
            n_estimators=200, learning_rate=0.05, max_depth=3, random_state=RANDOM_STATE
        ),
        "Extra Trees": ExtraTreesRegressor(
            n_estimators=250, min_samples_leaf=2, n_jobs=-1, random_state=RANDOM_STATE
        ),
    }

models = build_models()
training_times = {}
validation_predictions = {}
for model_name, model in models.items():
    start_time = perf_counter()
    model.fit(X_train, y_train)
    training_times[model_name] = perf_counter() - start_time
    validation_predictions[model_name] = model.predict(X_validation)
    print(f"{model_name}: trained in {training_times[model_name]:.2f} seconds")

---
## Section 6: Evaluate Models
Compare validation MAE, MSE, RMSE, and R². Rank primarily by higher R², then lower RMSE; the same untouched validation set is used for every baseline.

In [ ]:
def regression_metrics(actual, predicted):
    """Compute standard regression metrics as plain Python floats."""
    mse = mean_squared_error(actual, predicted)
    return {
        "MAE": float(mean_absolute_error(actual, predicted)),
        "MSE": float(mse),
        "RMSE": float(np.sqrt(mse)),
        "R2": float(r2_score(actual, predicted)),
    }

baseline_rows = []
for model_name, predictions in validation_predictions.items():
    baseline_rows.append(
        {"Model": model_name, "Training seconds": training_times[model_name],
         **regression_metrics(y_validation, predictions)}
    )
comparison_df = pd.DataFrame(baseline_rows).sort_values(
    ["R2", "RMSE"], ascending=[False, True]
).reset_index(drop=True)
display(comparison_df.style.format({"Training seconds": "{:.2f}", "MAE": "{:.3f}", "MSE": "{:.3f}", "RMSE": "{:.3f}", "R2": "{:.4f}"}))

---
## Section 7: Cross Validation
Run shuffled 5-fold cross-validation on training data only. Report mean and standard deviation of R²; no validation or test observations enter these folds.

In [ ]:
cv_strategy = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for model_name, model in models.items():
    scores = cross_val_score(
        clone(model), X_train, y_train, scoring="r2", cv=cv_strategy, n_jobs=1
    )
    cv_rows.append(
        {"Model": model_name, "Mean CV R2": scores.mean(),
         "Std CV R2": scores.std(ddof=1)}
    )
cv_results = pd.DataFrame(cv_rows).sort_values("Mean CV R2", ascending=False)
display(cv_results.style.format({"Mean CV R2": "{:.4f}", "Std CV R2": "{:.4f}"}))

---
## Section 8: Hyperparameter Tuning
Tune only the highest-ranked baseline with `RandomizedSearchCV`. Search cross-validation uses training rows only; validation R² decides whether the tuned candidate or its baseline is retained. Refit the selected configuration on train plus validation, then evaluate test once.

In [ ]:
best_baseline_name = comparison_df.iloc[0]["Model"]
parameter_spaces = {
    "Random Forest": {"n_estimators": [200, 300, 400], "max_depth": [None, 12, 20, 30], "min_samples_split": [2, 5, 10], "min_samples_leaf": [1, 2, 4], "max_features": [1.0, "sqrt"]},
    "XGBoost": {"n_estimators": [150, 250, 350], "max_depth": [3, 5, 7], "learning_rate": [0.03, 0.05, 0.1], "subsample": [0.8, 1.0], "colsample_bytree": [0.8, 1.0], "min_child_weight": [1, 3, 5]},
    "LightGBM": {"n_estimators": [150, 250, 350], "num_leaves": [15, 31, 63], "max_depth": [-1, 8, 12], "learning_rate": [0.03, 0.05, 0.1], "min_child_samples": [10, 20, 30], "subsample": [0.8, 1.0], "colsample_bytree": [0.8, 1.0]},
    "Gradient Boosting": {"n_estimators": [100, 200, 300], "learning_rate": [0.03, 0.05, 0.1], "max_depth": [2, 3, 4], "min_samples_split": [2, 5, 10], "min_samples_leaf": [1, 2, 4], "subsample": [0.8, 1.0], "max_features": [None, "sqrt"]},
    "Extra Trees": {"n_estimators": [200, 300, 400], "max_depth": [None, 12, 20, 30], "min_samples_split": [2, 5, 10], "min_samples_leaf": [1, 2, 4], "max_features": [1.0, "sqrt"]},
}

search = RandomizedSearchCV(
    estimator=clone(models[best_baseline_name]),
    param_distributions=parameter_spaces[best_baseline_name],
    n_iter=10, scoring="r2", cv=3, random_state=RANDOM_STATE,
    n_jobs=1, refit=True, error_score="raise"
)
tuning_start = perf_counter()
search.fit(X_train, y_train)
tuning_seconds = perf_counter() - tuning_start
tuned_model = search.best_estimator_
tuned_validation_predictions = tuned_model.predict(X_validation)
tuned_validation_metrics = regression_metrics(y_validation, tuned_validation_predictions)

baseline_metrics = comparison_df.set_index("Model").loc[best_baseline_name]
tuned_row = {"Model": f"{best_baseline_name} (tuned)", "Training seconds": tuning_seconds, **tuned_validation_metrics}
selection_table = pd.concat([comparison_df, pd.DataFrame([tuned_row])], ignore_index=True)
selection_table = selection_table.sort_values(["R2", "RMSE"], ascending=[False, True]).reset_index(drop=True)
tuned_is_better = (tuned_validation_metrics["R2"] > baseline_metrics["R2"] or (np.isclose(tuned_validation_metrics["R2"], baseline_metrics["R2"]) and tuned_validation_metrics["RMSE"] < baseline_metrics["RMSE"]))
best_model_name = f"{best_baseline_name} (tuned)" if tuned_is_better else best_baseline_name
selected_validation_model = tuned_model if tuned_is_better else models[best_baseline_name]

X_train_validation = pd.concat([X_train, X_validation], axis=0)
y_train_validation = pd.concat([y_train, y_validation], axis=0)
best_model = clone(selected_validation_model)
refit_start = perf_counter()
best_model.fit(X_train_validation, y_train_validation)
final_refit_seconds = perf_counter() - refit_start
test_predictions = best_model.predict(X_test)
test_metrics = regression_metrics(y_test, test_predictions)
validation_model = selected_validation_model
validation_final_predictions = validation_model.predict(X_validation)

print(f"Tuned baseline: {best_baseline_name}")
print(f"Search time: {tuning_seconds:.2f} seconds; best CV R2: {search.best_score_:.4f}")
print(f"Best parameters: {search.best_params_}")
print(f"Selected model: {best_model_name}")
print(f"Final refit time: {final_refit_seconds:.2f} seconds")
print(f"Held-out test metrics: {test_metrics}")

---
## Section 9: Feature Importance
Tree-based feature importance indicates how strongly the selected model used each input while fitting. The leading signals can reflect real maintenance risk patterns, but importance is not causal evidence and correlated features can share or obscure importance. Validate these patterns with domain experts before operational use.

In [ ]:
if not hasattr(best_model, "feature_importances_"):
    raise AttributeError("Selected estimator does not expose tree feature_importances_.")
importance_df = pd.DataFrame({"Feature": feature_columns, "Importance": best_model.feature_importances_})
top_features = importance_df.nlargest(20, "Importance").sort_values("Importance")
fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(data=top_features, x="Importance", y="Feature", color="#287C8E", ax=ax)
ax.set(title=f"Top 20 Feature Importances - {best_model_name}", xlabel="Importance", ylabel="")
fig.tight_layout()
importance_path = FIGURES_DIR / "feature_importance_top20.png"
fig.savefig(importance_path, dpi=160, bbox_inches="tight")
plt.show()
print(f"Saved: {importance_path}")
display(top_features.sort_values("Importance", ascending=False).reset_index(drop=True))

---
## Section 10: Prediction Analysis
Inspect the selected train-only validation estimator on validation data and the final train-plus-validation refit on the untouched test set. For each split, save actual-versus-predicted, prediction-error histogram, and residual plots.

In [ ]:
def save_prediction_diagnostics(actual, predicted, split_name):
    """Save actual/predicted, error-distribution, and residual diagnostics."""
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    errors = predicted - actual
    safe_name = split_name.lower()

    fig, ax = plt.subplots(figsize=(7, 6))
    sns.scatterplot(x=actual, y=predicted, alpha=0.45, s=24, ax=ax)
    lower = min(actual.min(), predicted.min())
    upper = max(actual.max(), predicted.max())
    ax.plot([lower, upper], [lower, upper], linestyle="--", color="#C44E52")
    ax.set(title=f"{split_name}: Actual vs Predicted", xlabel="Actual priority_score", ylabel="Predicted priority_score")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f"{safe_name}_actual_vs_predicted.png", dpi=160, bbox_inches="tight")
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 5))
    sns.histplot(errors, bins=35, kde=True, color="#287C8E", ax=ax)
    ax.axvline(0, color="#C44E52", linestyle="--")
    ax.set(title=f"{split_name}: Prediction Error Distribution", xlabel="Prediction error (predicted - actual)", ylabel="Count")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f"{safe_name}_error_histogram.png", dpi=160, bbox_inches="tight")
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 5))
    sns.scatterplot(x=predicted, y=errors, alpha=0.45, s=24, ax=ax)
    ax.axhline(0, color="#C44E52", linestyle="--")
    ax.set(title=f"{split_name}: Residual Plot", xlabel="Predicted priority_score", ylabel="Residual (predicted - actual)")
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f"{safe_name}_residuals.png", dpi=160, bbox_inches="tight")
    plt.show()
    print(f"Saved {split_name} diagnostic figures in {FIGURES_DIR}")

save_prediction_diagnostics(y_validation, validation_final_predictions, "Validation")
save_prediction_diagnostics(y_test, test_predictions, "Test")

---
## Section 11: Model Selection
Compare every baseline with the tuned candidate using validation performance. Select by highest validation R² and use validation RMSE to break ties. The test metrics are reported only for the selected refit as a final generalization check; they do not influence selection.

In [ ]:
selection_table["Selected"] = selection_table["Model"].eq(best_model_name)
display(
    selection_table.style.format(
        {
            "Training seconds": "{:.2f}",
            "MAE": "{:.3f}",
            "MSE": "{:.3f}",
            "RMSE": "{:.3f}",
            "R2": "{:.4f}",
        }
    )
)
print(f"Selected model: {best_model_name}")

if tuned_is_better:
    reason = (
        "The tuned {} achieved validation R2 of {:.4f} and RMSE of {:.3f}, "
        "outperforming its baseline under the stated selection rule."
    ).format(
        best_baseline_name,
        tuned_validation_metrics["R2"],
        tuned_validation_metrics["RMSE"],
    )
else:
    reason = (
        "The {} baseline retained better validation performance "
        "(R2 {:.4f}, RMSE {:.3f}) than its tuned candidate under the "
        "stated selection rule."
    ).format(
        best_baseline_name,
        baseline_metrics["R2"],
        baseline_metrics["RMSE"],
    )

print(reason)
print(f"Final held-out test metrics: {test_metrics}")

---
## Section 12: Save Best Model
Persist the final estimator and its exact feature order so deployment code can construct compatible input matrices.

In [ ]:
best_model_path = MODELS_DIR / "best_model.pkl"
feature_columns_path = MODELS_DIR / "feature_columns.pkl"
joblib.dump(best_model, best_model_path)
joblib.dump(feature_columns, feature_columns_path)
print(f"Saved model: {best_model_path}")
print(f"Saved feature schema: {feature_columns_path}")

---
## Section 13: Training Report
Summarize the training runs, validation comparison, final test evaluation, selection rationale, and practical next steps. No scheduling optimization is performed here.

In [ ]:
training_report = pd.DataFrame(
    [
        {"Item": "Models trained", "Details": ", ".join(models.keys()) + f", tuned {best_baseline_name}"},
        {"Item": "Baseline training times (seconds)", "Details": str({name: round(seconds, 2) for name, seconds in training_times.items()})},
        {"Item": "Hyperparameter search time (seconds)", "Details": round(tuning_seconds, 2)},
        {"Item": "Final refit time (seconds)", "Details": round(final_refit_seconds, 2)},
        {"Item": "Best model", "Details": best_model_name},
        {"Item": "Selection reason", "Details": reason},
        {"Item": "Final test metrics", "Details": str({key: round(value, 4) for key, value in test_metrics.items()})},
        {"Item": "Potential improvements", "Details": "Collect time-based and route-grouped holdouts; calibrate uncertainty; audit subgroup errors with railway engineers; expand tuning only after a stable baseline."},
    ]
)
display(training_report)

comparison_path = REPORTS_DIR / "model_comparison.csv"
report_path = REPORTS_DIR / "model_training_report.csv"
selection_table.to_csv(comparison_path, index=False)
training_report.to_csv(report_path, index=False)
print(f"Saved comparison: {comparison_path}")
print(f"Saved training report: {report_path}")
print("Training stage complete. The selected model is ready for deployment; no optimization was run.")